# Week 8 skills notebook: version control and sharing work

This notebook belongs to the skills track. It follows a made-up team of two analysts, Ana and Ben, who count passengers on a made-up bus line.

You will run real git commands in a practice folder that the notebook creates, and removes again at the end. Everything you see is what git prints on a laptop.

**How to use it:** run each cell in order. Read every `git status`, because it tells you the next command. If you restart the notebook, run it again from the top.

## 1. A practice folder and a helper

Git is a program that runs on its own, outside Python. The next cell makes a temporary practice folder and a small function, `git()`, that runs one git command in a folder and prints git's answer, the way a terminal would.

The function also leaves out any git settings on your own computer, so every run of this notebook behaves the same.

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

workspace = tempfile.TemporaryDirectory(prefix="git-practice-")
root = Path(workspace.name)          # removed again in the last section
ana = root / "bus-counts"            # Ana's project folder
ana.mkdir()

# leave out the git settings of this computer, so every run behaves the same
GIT_ENV = dict(os.environ, GIT_CONFIG_GLOBAL=os.devnull, GIT_CONFIG_NOSYSTEM="1")

def git(*args, where=ana, show=True):
    '''Run one git command inside a folder, print what git answers, and return it as text.'''
    result = subprocess.run(["git", *args], cwd=where, env=GIT_ENV, capture_output=True, text=True)
    answer = (result.stdout + result.stderr).strip()
    if show:
        print("$ git", " ".join(args))
        if answer:
            print(answer)
        print()
    return answer

git("--version")

## 2. Start a repository

A repository is a folder with a history attached. `git init` starts the history, and `-b main` names the main line of history `main`.

Git signs every commit with a name and an email address, so the next cell sets both for this folder. On your own laptop you set them once for all projects with `git config --global`. For work on GitHub, use the private *noreply* address that GitHub offers under Settings, then Emails, so your own address never appears in a public history.

In [ ]:
git("init", "-b", "main")
git("config", "user.name", "Ana")
git("config", "user.email", "ana@example.org")
git("status")

`git status` says there are no commits yet and nothing to commit. The folder is empty, and git is watching it.

## 3. The first commit

Ana's first file holds a passenger count for each day over twelve weeks. On some days the counter was switched off, so the count is blank. On a few days the counter was broken and recorded zero.

In [ ]:
rng = np.random.default_rng(8)
days = pd.date_range("2026-03-02", periods=84, freq="D")
passengers = rng.poisson(380, size=84).astype(float)
passengers[[9, 30, 31, 58]] = np.nan      # the counter was switched off
passengers[[44, 45, 70]] = 0              # the counter was broken and recorded zero
pd.DataFrame({"date": days.date, "passengers": passengers}).to_csv(ana / "passengers.csv", index=False)

git("status")

The file is *untracked*: it is on disk, and git does not know it yet. Git's own text names the next command, `git add`.

Adding puts the file's current state on the list for the next commit. That list is called the *staging area*.

In [ ]:
git("add", "passengers.csv")
git("status")

The file is now under *Changes to be committed*, and the history is still empty. `git commit` turns everything staged into one snapshot with a message.

In [ ]:
git("commit", "-m", "Add twelve weeks of daily passenger counts")
git("log", "--oneline")

The log has one line. The seven characters at its start are the short form of the commit's name. Git computes that name from the files, the message, the author, the time and the commit before it, so a change to any of them gives a different name.

## 4. Code and its result

Ana writes a short script, `analysis.py`, that computes the average number of passengers per day and writes it to `results.txt`. The next cell writes the script into the folder, runs it the way a terminal would, and commits the script together with its result.

In [ ]:
ANALYSIS = '''import pandas as pd

counts = pd.read_csv("passengers.csv")
used = counts.fillna(0)
average = used["passengers"].mean()

with open("results.txt", "w") as f:
    print("Days used:", len(used), file=f)
    print(f"Average passengers per day: {average:.1f}", file=f)
'''
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
print((ana / "results.txt").read_text())

git("add", "analysis.py", "results.txt")
git("commit", "-m", "Compute the average passengers per day")

## 5. A README a stranger can follow

GitHub shows a file called `README.md` on the front page of every repository, so every visitor reads it first. The `.md` stands for Markdown, a way of writing plain text that turns into formatted text. A line that starts with `#` is a heading, a line that starts with `-` is a list item, and text between backticks shows as code.

A README a stranger can follow answers five questions: what is this, where did the data come from, how do I run it, what is in the folder, and what is its status?

In [ ]:
README = '''# Bus line passenger counts

Daily passenger counts for one bus line over twelve weeks, and the average per day.

## The data
`passengers.csv` holds one row per day. A blank count means the counter was switched off.

## How to run it
Run `python analysis.py`. It reads `passengers.csv` and writes `results.txt`.

## What is in the folder
- `passengers.csv`: the daily counts
- `analysis.py`: the analysis
- `results.txt`: the average it produced

## Status
Status: first analysis done.
'''
(ana / "README.md").write_text(README)
git("add", "README.md")
git("commit", "-m", "Write a README for a first-time reader")
git("log", "--oneline")

## 6. Change the analysis and read the diff

Ana notices that `fillna(0)` counts the days the counter was switched off as days with no passengers. She changes one line and runs the script again. Before she commits, `git diff` shows exactly what changed.

In [ ]:
ANALYSIS = ANALYSIS.replace("used = counts.fillna(0)", 'used = counts.dropna(subset=["passengers"])')
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
git("status")
git("diff")

`git diff` printed one block for each changed file. A line that starts with `-` was removed, and a line that starts with `+` was added, so an edited line appears as one of each.

The line between the `@@` marks is the *hunk header*. Take `@@ -12,4 +12,5 @@` as an example. The block starts at line 12 in both files. It runs for four lines in the old file and for five in the new one.

The commit below has a subject and a body. The second `-m` adds the body, which says why.

In [ ]:
V2_SUBJECT = "Skip days with no count in the average"
git("add", "analysis.py", "results.txt")
git("commit", "-m", V2_SUBJECT,
    "-m", "A blank means the counter was switched off, not that nobody rode the bus.")
git("log", "--oneline")

## 7. Commit messages that record why

The history is only as useful as its messages, and six months from now you are the stranger reading them. Four rules make a subject useful.

- It has 50 characters or fewer, so it fits on one line of the log.
- It is written as a command, because it completes the sentence *If applied, this commit will ...*. Write *Skip days with no count*, and avoid *Skipped days*.
- It names the change in words a reader understands. *Update README.md* and *fix stuff* tell the reader nothing.
- When the reason is not obvious, a body after the subject says why.

The function below checks the first three rules. Whether the message is true is yours to check.

In [ ]:
VAGUE = {"update", "updates", "updated", "fix", "fixes", "fixed", "change", "changes", "stuff",
         "wip", "misc", "final", "version", "readme", "readme.md", "the", "and", "of",
         "wijzigingen", "aanpassingen", "dingen", "gefikst", "definitieve", "versie", "de", "het", "en"}
PAST = {"added", "fixed", "updated", "changed", "removed", "deleted", "wrote", "made",
        "toegevoegd", "aangepast", "verwijderd", "gewijzigd", "opgelost", "gemaakt", "geschreven"}

def check_subject(subject):
    '''List the rules a commit subject breaks. An empty list means it passes.'''
    words = subject.split()
    problems = []
    if len(subject) > 50:
        problems.append("longer than 50 characters")
    if not subject[:1].isupper():
        problems.append("does not start with a capital letter")
    if subject.endswith("."):
        problems.append("ends with a full stop")
    if words and (words[0].lower() in PAST or words[-1].lower().strip(".") in PAST):
        problems.append("past tense instead of a command")
    if len(words) < 3 or all(w.lower().strip(".,!") in VAGUE for w in words):
        problems.append("too vague to say what changed")
    return problems

for subject in ["Update README.md", "Fixed stuff", "fixing the average.", V2_SUBJECT,
                "Skip blank days in the average and give the output file a clearer name"]:
    print(repr(subject), "->", check_subject(subject) or "passes")

### Your turn 1

The list `bad` holds five weak subjects from a made-up history. The comment beside each one says what that commit actually changed. Write a better subject for each one in the list `better`, in the same order.

Ask your AI assistant for suggestions if you like, then test each suggestion with `check_subject`. Assistants often write *Updated* or end a subject with a full stop.

In [ ]:
bad = [
    "Update README.md",     # added the section that explains how to run the analysis
    "fixed stuff",          # made pandas read the date column as dates instead of text
    "Changes",              # removed a row that appeared twice for 14 March
    "Added a chart.",       # added a chart of the average passengers per week
    "final version",        # wrote the results for the transport office
]

In [ ]:
# your turn 1: write five better subjects, in the same order as bad
better = [
    "",
    "",
    "",
    "",
    "",
]

In [ ]:
# CHECK
assert len(better) == len(bad), "write one subject for each of the five"
for old, new in zip(bad, better):
    assert new.strip() and new != old, f"rewrite {old!r}"
    assert check_subject(new) == [], f"{new!r}: {check_subject(new)}"
print("CHECK passed: all", len(better), "subjects follow the rules")

## 8. Which version produced this number?

A result changes every time the analysis changes. When a report quotes a number, someone will ask which version of the analysis produced it. The history can answer, because each result was committed together with the code that made it.

The next cell writes a short report that quotes the current average, and commits it. Then the analysis changes once more: the depot tells Ana that the counter was broken on the days it recorded zero.

In [ ]:
average_line = (ana / "results.txt").read_text().splitlines()[-1]
(ana / "report.md").write_text("# Report for the transport office\n\n" + average_line + "\n")
git("add", "report.md")
git("commit", "-m", "Write the first report for the transport office")

V3_SUBJECT = "Leave out days the counter recorded zero"
ANALYSIS = ANALYSIS.replace('used = counts.dropna(subset=["passengers"])',
                            'used = counts.dropna(subset=["passengers"])\nused = used[used["passengers"] > 0]')
(ana / "analysis.py").write_text(ANALYSIS)
subprocess.run([sys.executable, "analysis.py"], cwd=ana, check=True)
git("add", "analysis.py", "results.txt")
git("commit", "-m", V3_SUBJECT,
    "-m", "The depot says the counter was broken on those days, so a zero there is not a count.")

print("report.md says:  ", (ana / "report.md").read_text().splitlines()[-1])
print("results.txt says:", (ana / "results.txt").read_text().splitlines()[-1])

The report and the results now disagree, and neither file says why. The history does.

`git log -- results.txt` lists only the commits that changed `results.txt`. `git show <commit>:results.txt` prints the file as it was in one of them. The cell below prints it for the oldest one.

In [ ]:
git("log", "--oneline", "--", "results.txt")
oldest = git("log", "--format=%h", "--", "results.txt", show=False).split()[-1]
git("show", f"{oldest}:results.txt")

### Your turn 2

Find the commit whose `results.txt` holds the line that `report.md` quotes. Store the commit's short name in `found` and its subject in `found_subject`.

Call `git(..., show=False)` to get git's answer as text without printing it, and loop over the commits that changed `results.txt`.

In [ ]:
# your turn 2: find the commit that produced the number in report.md
found = None
found_subject = None

In [ ]:
# CHECK
assert found is not None, "found is still empty"
assert found_subject == V2_SUBJECT, f"{found_subject!r} did not produce the number in report.md"
print("CHECK passed: the report quotes the result of", repr(found_subject))

A tag gives one commit a name, so the next reader does not have to search. The cell below tags the commit that produced the report.

In [ ]:
git("tag", "report-1", found)
git("log", "--oneline", "--decorate")

## 9. Try an idea on a branch

A branch is a name that points at a commit, and `main` is the default one. When you commit on a new branch, its name moves forward and `main` stays where it was. A half-finished idea then never reaches the version your team relies on. `git switch -c` makes a branch and moves to it.

In [ ]:
git("switch", "-c", "weekly-averages")
counts = pd.read_csv(ana / "passengers.csv", parse_dates=["date"])
counts = counts[counts["passengers"] > 0]          # the same days that analysis.py uses
weekly = counts.resample("W", on="date")["passengers"].mean().round(1)
weekly.to_csv(ana / "weekly.csv")
git("add", "weekly.csv")
git("commit", "-m", "Add the average passengers per week")
print("files on weekly-averages:", sorted(p.name for p in ana.iterdir() if not p.name.startswith(".")))
print()
git("switch", "main")
print("files on main:", sorted(p.name for p in ana.iterdir() if not p.name.startswith(".")))

`weekly.csv` disappeared from the folder when Ana switched back. Git did not delete it: `main` points at a commit that never had the file.

`git merge` brings the branch's commits into `main`. Because `main` had no new commits of its own, git only moves the name forward, which is called a *fast-forward*. After the merge, the branch name has done its job and can go.

In [ ]:
git("merge", "weekly-averages")
git("branch", "-d", "weekly-averages")
git("log", "--oneline", "--graph")

## 10. Share the work with a teammate

On a team, the shared copy of a repository usually lives on GitHub. Git calls it a *remote*, and the usual name for it is `origin`. Here a second folder on this machine plays GitHub's part, so you can watch both sides at once.

`git push` sends your new commits to the remote. Ben then clones the repository, which copies the whole history into his own folder. Each of them also tells git to merge when it pulls.

In [ ]:
shared = root / "shared.git"          # plays the part of the repository on GitHub
git("init", "--bare", "-b", "main", str(shared), where=root)
git("remote", "add", "origin", str(shared))
git("config", "pull.rebase", "false")             # when pulling, merge
git("push", "-u", "origin", "main", "--tags")

ben = root / "ben-copy"
git("clone", str(shared), str(ben), where=root)
git("config", "user.name", "Ben", where=ben)
git("config", "user.email", "ben@example.org", where=ben)
git("config", "pull.rebase", "false", where=ben)
git("log", "--oneline", where=ben)

### Two people change one line

Ana and Ben both edit the status line of the README. Ben pushes first.

In [ ]:
def set_status(folder, text):
    '''Replace the README's status line in one person's folder.'''
    path = folder / "README.md"
    lines = ["Status: " + text if line.startswith("Status:") else line
             for line in path.read_text().splitlines()]
    path.write_text("\n".join(lines) + "\n")

set_status(ben, "waiting for the depot's list of broken counter days.")
git("add", "README.md", where=ben)
git("commit", "-m", "Note that we wait for the depot's list", where=ben)
git("push", where=ben)

set_status(ana, "report sent to the transport office.")
git("add", "README.md")
git("commit", "-m", "Record that the report was sent")
git("push")

Git refused Ana's push. Read its hint: the remote holds work that Ana does not have, and git suggests that she bring it in first with `git pull`.

Pulling brings down Ben's commit and merges it with Ana's. Both of them changed the same line, so git cannot know which text is right. It merges everything else, stops at that line and writes both versions into the file.

In [ ]:
git("pull")
print((ana / "README.md").read_text())

The lines between `<<<<<<< HEAD` and `=======` are Ana's side, and the lines between `=======` and `>>>>>>>` are Ben's.

Resolving a conflict means reading both sides, writing what the line should say, deleting the three marker lines and committing. Here both statements are true, so the resolved line keeps both facts.

In [ ]:
text = (ana / "README.md").read_text()
start = text.index("<<<<<<<")
end = text.index("\n", text.index(">>>>>>>")) + 1
resolved = "Status: report sent; waiting for the depot's list of broken counter days.\n"
(ana / "README.md").write_text(text[:start] + resolved + text[end:])

git("add", "README.md")
git("commit", "--no-edit")
git("push")
git("pull", where=ben)
git("log", "--oneline", "--graph", "-5", where=ben)

Ben's log shows the merge: two lines of history that join at a commit with two parents. That is the safe answer to every rejected push: pull, resolve any conflict, commit, then push.

## 11. When the AI is wrong: a command that erases a teammate's work

Ben adds the depot's list of broken counter days and pushes. Ana has not pulled since. She improves one line of the README, commits it and tries to push, and git refuses as before.

This time she pastes the error into an AI assistant and asks how to make the push work.

In [ ]:
counts = pd.read_csv(ben / "passengers.csv")
counts[counts["passengers"] == 0][["date"]].to_csv(ben / "depot-list.csv", index=False)
git("add", "depot-list.csv", where=ben)
git("commit", "-m", "Add the depot's list of broken counter days", where=ben)
git("push", where=ben)
ben_commit = git("rev-parse", "HEAD", where=ben, show=False)

readme = (ana / "README.md").read_text()
(ana / "README.md").write_text(readme.replace("Run `python analysis.py`.",
                                              "Run `python analysis.py` from inside this folder."))
git("add", "README.md")
git("commit", "-m", "Say which folder to run the analysis from")
git("push")
shared_before = git("rev-parse", "main", where=shared, show=False)

The assistant answered: *The remote contains changes you do not have. To overwrite them, run `git push --force`.*

The next cell runs exactly what it suggested. Read the cell after it before you decide whether the advice was good.

In [ ]:
# Suggested by an AI assistant
git("push", "--force")

In [ ]:
def in_shared_history(commit):
    '''True when the commit is part of main in the shared repository.'''
    check = subprocess.run(["git", "merge-base", "--is-ancestor", commit, "main"],
                           cwd=shared, env=GIT_ENV, capture_output=True)
    return check.returncode == 0

erased = git("rev-list", "--count", shared_before, "--not", "main", where=shared, show=False)
print("Commits erased from the shared history:", erased)
print("Ben's commit is still in the shared history:", in_shared_history(ben_commit))

The force push replaced the shared history with Ana's copy. Ben's commit is gone from it, and the depot's list with it. Git's refusal had named the safe path, which was to pull first, and the assistant's answer skipped that step. A force push rewrites history that other people rely on, so on a shared branch it is almost never the right answer.

The work can still be saved here, because Ben's own folder still holds his commit. Ben pulls, which merges Ana's version with his, and pushes again.

In [ ]:
git("pull", "--no-edit", where=ben)
git("push", where=ben)
print("Ben's commit is back in the shared history:", in_shared_history(ben_commit))
print()
git("pull")

The rescue depended on luck. Had Ben deleted his folder, or run the same command himself, his commit would be gone for good.

The right answer to a rejected push is the one from section 10: pull, resolve, then push. Whenever an assistant gives you a git command, ask it what the command will erase before you run it.

## 12. When the AI is wrong: a commit that publishes a password

Ana's folder now holds two more files that must never be committed. `.env` keeps the password to the counter database, and `raw_counter_export.csv` is a large raw export from the counters.

She asks an AI assistant: *How do I save all my work and share it with the team?*

In [ ]:
(ana / ".env").write_text("COUNTER_DB_PASSWORD=made-up-password-for-practice\n")
pd.DataFrame({"reading": rng.integers(0, 1_000_000, size=300_000)}).to_csv(
    ana / "raw_counter_export.csv", index=False)

def size_mb(path):
    '''Size of a file, or of everything in a folder, in megabytes.'''
    files = [path] if path.is_file() else [p for p in path.rglob("*") if p.is_file()]
    return round(sum(p.stat().st_size for p in files) / 1e6, 1)

print("Size of the raw export in MB:", size_mb(ana / "raw_counter_export.csv"))
print("Size of the stored history before, in MB:", size_mb(ana / ".git" / "objects"))
print()
git("status")

The next cell runs the assistant's answer exactly as written.

In [ ]:
# Suggested by an AI assistant
git("add", ".")
git("commit", "-m", "update")
git("push")

`git add .` staged everything in the folder that git was not told to ignore. Look at what the commit holds, then at Ben's folder after he pulls.

In [ ]:
git("show", "--stat", "--format=%s", "HEAD")
git("pull", where=ben)
print("Ben's copy of .env:", (ben / ".env").read_text().strip())

Ana tries to undo it. She tells git to stop tracking both files, writes a `.gitignore` that lists them, commits and pushes. `git rm --cached` takes a file out of the next commit and leaves it on the disk.

In [ ]:
git("rm", "--cached", ".env", "raw_counter_export.csv")
(ana / ".gitignore").write_text(".env\nraw_counter_export.csv\n")
git("add", ".gitignore")
git("commit", "-m", "Stop tracking the password file and raw export")
git("push")
git("ls-files")

The two files are gone from the newest commit. The commit before it still holds them.

In [ ]:
leaked = git("show", "HEAD~1:.env", show=False)
print("Password still readable from the history:", "PASSWORD" in leaked)
print(leaked)
print("Size of the stored history after the removal, in MB:", size_mb(ana / ".git" / "objects"))

A later commit never removes a file from the commits before it. Ben holds the password now, and on a public repository anyone who looked may hold it too. The only safe response to a pushed secret is to treat it as leaked and change it at once. Removing it from the history takes special tools and every teammate's help, and it cannot reach copies already made. The raw export also stays in the history, so every future clone downloads it.

Two habits prevent both problems. Write the `.gitignore` before the first `git add`, and read `git status` before every commit.

### Your turn 3: a clean start

The next cell makes a new folder, `clean-start`, with the same files, the password file and the raw export among them, and starts a repository in it.

Make a first commit that holds only the files that belong there. Write a `.gitignore` first, stage the right files, read `git status`, and commit with a subject that passes `check_subject`. Pass `where=clean` to every `git()` call.

In [ ]:
clean = root / "clean-start"
clean.mkdir()
for name in ["README.md", "analysis.py", "passengers.csv", "results.txt", ".env", "raw_counter_export.csv"]:
    shutil.copy(ana / name, clean / name)
git("init", "-b", "main", where=clean)
git("config", "user.name", "Ana", where=clean)
git("config", "user.email", "ana@example.org", where=clean)
git("status", where=clean)

In [ ]:
# your turn 3: write clean / ".gitignore", stage the right files and commit, always with where=clean

In [ ]:
# CHECK
tracked = set(git("ls-tree", "-r", "--name-only", "HEAD", where=clean, show=False).split())
assert ".env" not in tracked, "the password file is in the commit"
assert "raw_counter_export.csv" not in tracked, "the raw export is in the commit"
assert tracked == {".gitignore", "README.md", "analysis.py", "passengers.csv", "results.txt"}, tracked
subject = git("log", "-1", "--format=%s", where=clean, show=False)
assert check_subject(subject) == [], f"{subject!r}: {check_subject(subject)}"
assert "!! .env" in git("status", "--ignored", "--porcelain", where=clean, show=False)
print("CHECK passed: the first commit holds", len(tracked), "files, and git ignores the password file")

## 13. The same commands on GitHub

Everything above works the same way with a repository on GitHub. Three things change.

- The remote is a web address such as `https://github.com/your-team/your-repository.git`, and `git clone` takes that address.
- Pushing asks for a *personal access token* in place of your password. Make a fine-grained token for one repository with a short expiry, and type it only into a hidden prompt such as Python's `getpass`. A token typed into a cell, or into the remote's address, is saved where others can find it.
- Anyone can read a public repository, so the `.gitignore` and the `git status` habit matter even more there.

## 14. Clean up

The last cell removes the practice folder and everything in it.

In [ ]:
workspace.cleanup()
print("Practice folder removed:", not root.exists())

## What you did

You started a repository, staged and committed files, read a diff, and wrote messages that say why. You found the version that produced a number and tagged it, merged a branch, shared work through a remote and resolved a conflict.

You also watched two AI suggestions do damage. A force push erased a teammate's commit, and a `git add .` published a password. Most of the rest of git is a variation on the commands in this notebook.